In [6]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
from tqdm import tqdm

In [7]:
load_dotenv()
OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY")

In [72]:
# Date range for the project
START_DATE = "2016-01-01"
END_DATE   = "2025-12-31"

DATA_DIR = "../data"
DATA_DIR_RAW = "../data/raw"
DATA_DIR_PROCESSED = "../data/processed"
FIGURES_DIR = "../data/figures"

os.makedirs(DATA_DIR,       exist_ok=True)
os.makedirs(DATA_DIR_RAW,       exist_ok=True)
os.makedirs(DATA_DIR_PROCESSED, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

#### Collection of PM25 and PM10 Data from Open AQ API

In [9]:
# Delhi bounding box of latitute and longitude
DELHI_BOUNDS = {
    "lat_min": 28.40,
    "lat_max": 28.88,
    "lon_min": 76.84,
    "lon_max": 77.35
}

headers = {"X-API-Key": OPENAQ_API_KEY}
stations = []
page = 1

while True:
    url = "https://api.openaq.org/v3/locations"
    params = {
        "bbox":     f"{DELHI_BOUNDS['lon_min']},{DELHI_BOUNDS['lat_min']},{DELHI_BOUNDS['lon_max']},{DELHI_BOUNDS['lat_max']}",
        "limit":    100,
        "page":     page
    }

    response = requests.get(url, params=params, headers=headers, timeout=30)
    data = response.json()
    results = data.get("results", [])
    
    if not results:
        break

    for loc in results:
        for sensor in loc.get("sensors", []):
            param_name = sensor["parameter"]["name"]
            if param_name in ["pm25", "pm10"]:
                stations.append({
                    "location_id":   loc["id"],
                    "location_name": loc["name"],
                    "sensor_id":     sensor["id"],
                    "parameter":     param_name,
                    "lat":           loc.get("coordinates", {}).get("latitude"),
                    "lon":           loc.get("coordinates", {}).get("longitude"),
                })
    
    # Stop if we've got all pages
    found = data["meta"].get("found", "0")
    total = int(found) if found != ">1000" else 1000
    if page * 100 >= total:
        break
    page += 1

stations_df = pd.DataFrame(stations).drop_duplicates(subset=["location_id", "parameter"])
print(f"Found {len(stations_df)} station-parameter combos in Delhi")


Found 165 station-parameter combos in Delhi


In [10]:
stations_df.head()

,location_id,location_name,sensor_id,parameter,lat,lon
0,13,"Delhi Technological University, Delhi - CPCB",13864,pm25,28.7440,77.1200
1,15,IGI Airport,26,pm10,28.5600,77.0940
2,15,IGI Airport,30,pm25,28.5600,77.0940
3,16,Civil Lines,31,pm10,28.6787,77.2262
4,16,Civil Lines,34,pm25,28.6787,77.2262


In [11]:
print(stations_df[["location_name", "parameter", "lat", "lon"]][:5].to_string())

                                  location_name parameter      lat      lon
0  Delhi Technological University, Delhi - CPCB      pm25  28.7440  77.1200
1                                   IGI Airport      pm10  28.5600  77.0940
2                                   IGI Airport      pm25  28.5600  77.0940
3                                   Civil Lines      pm10  28.6787  77.2262
4                                   Civil Lines      pm25  28.6787  77.2262


In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed

#Fetch the data from the stations
def fetch_sensor_daily_paginated(row):
    sensor_id     = row["sensor_id"]
    location_id   = row["location_id"]
    location_name = row["location_name"]
    parameter     = row["parameter"]

    url     = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"
    headers = {"X-API-Key": OPENAQ_API_KEY}
    all_rows = []
    page = 1

    while True:
        params = {
            "date_from": START_DATE,
            "date_to":   END_DATE,
            "limit":     1000,
            "page":      page,
        }
        try:
            resp    = requests.get(url, params=params, headers=headers, timeout=30)
            body    = resp.json()
            results = body.get("results", [])
            if not results:
                break
            for r in results:
                all_rows.append({
                    "date":          r["period"]["datetimeFrom"]["utc"][:10],
                    "value":         r["value"],
                    "sensor_id":     sensor_id,
                    "location_id":   location_id,
                    "location_name": location_name,
                    "parameter":     parameter,
                })
            # Stop if this page had fewer than 1000 — means last page
            if len(results) < 1000:
                break
            page += 1
        except Exception as e:
            print(f"⚠️ sensor {sensor_id} page {page}: {e}")
            break

    return all_rows

# Parallel fetch using multithreading
all_measurements = []
rows_list = [row for _, row in stations_df.iterrows()]

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_sensor_daily_paginated, row): row for row in rows_list}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching"):
        try:
            all_measurements.extend(future.result())
        except Exception as e:
            print(f"⚠️ Error: {e}")

aq_raw = pd.DataFrame(all_measurements)
aq_raw["date"] = pd.to_datetime(aq_raw["date"])

print(f"Total rows: {len(aq_raw)}")
print(f"Range: {aq_raw['date'].min()} → {aq_raw['date'].max()}")
print(f"Unique sensors: {aq_raw['sensor_id'].nunique()}")
print(f"Duplicates: {aq_raw.duplicated(subset=['date','sensor_id']).sum()}")


Fetching: 100%|██████████| 165/165 [00:05<00:00, 28.24it/s]


Total rows: 21942
Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
Unique sensors: 50
Duplicates: 0


In [13]:
# City-wide daily average across all stations per parameter
aq_daily = (
    aq_raw
    .groupby(["date", "parameter"])["value"]
    .mean()
    .reset_index()
    .pivot(index="date", columns="parameter", values="value")
    .reset_index()
)
aq_daily.columns.name = None
aq_daily.rename(columns={"pm25": "pm25_avg", "pm10": "pm10_avg"}, inplace=True)
aq_daily["date"] = pd.to_datetime(aq_daily["date"])

aq_daily.to_csv(f"{DATA_DIR_RAW}/air_quality_daily.csv", index=False)

print(f"City-level daily shape: {aq_daily.shape}")
print(f"Range: {aq_daily['date'].min()} → {aq_daily['date'].max()}")
print(f"Missing pm25: {aq_daily['pm25_avg'].isna().sum()} days")
print(f"Missing pm10: {aq_daily['pm10_avg'].isna().sum()} days")
aq_daily.head(10)

City-level daily shape: (2233, 3)
Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
Missing pm25: 1 days
Missing pm10: 56 days


,date,pm10_avg,pm25_avg
0,2016-01-29,NaN,356.000000
1,2016-01-30,NaN,203.000000
2,2016-01-31,NaN,124.000000
3,2016-02-01,NaN,136.000000
4,2016-02-02,NaN,148.000000
5,2016-02-03,NaN,137.000000
6,2016-02-04,425.0,225.666667
7,2016-02-05,366.0,211.000000
8,2016-02-06,227.0,160.000000
9,2016-02-07,284.5,132.733333


#### Fetch Weather Data from Open Meteo API


In [14]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [15]:
cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
om = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":   28.6139,
    "longitude":  77.2090,
    "start_date": START_DATE,
    "end_date":   END_DATE,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "wind_speed_10m_max",
        "wind_speed_10m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "visibility_mean",
    ],
    "timezone": "Asia/Kolkata"
}

responses = om.weather_api(url, params=params)
response  = responses[0]
daily     = response.Daily()

weather_df = pd.DataFrame({
    "date":           pd.date_range(
                          start=pd.to_datetime(daily.Time(),    unit="s", utc=True).tz_localize(None),
                          end=pd.to_datetime(daily.TimeEnd(),   unit="s", utc=True).tz_localize(None),
                          freq=pd.Timedelta(seconds=daily.Interval()),
                          inclusive="left"
                      ),
    "temp_max":       daily.Variables(0).ValuesAsNumpy(),
    "temp_min":       daily.Variables(1).ValuesAsNumpy(),
    "temp_mean":      daily.Variables(2).ValuesAsNumpy(),
    "wind_speed_max": daily.Variables(3).ValuesAsNumpy(),
    "wind_speed_mean":daily.Variables(4).ValuesAsNumpy(),
    "humidity_mean":  daily.Variables(5).ValuesAsNumpy(),
    "precipitation":  daily.Variables(6).ValuesAsNumpy(),
    "visibility":     daily.Variables(7).ValuesAsNumpy(),
})

weather_df["date"] = pd.to_datetime(weather_df["date"])


print(f"Weather data shape: {weather_df.shape}")
print(f"Range: {weather_df['date'].min()} → {weather_df['date'].max()}")
print(f"Missing values:\n{weather_df.isna().sum()}")
weather_df.head()

Weather data shape: (3653, 9)
Range: 2015-12-31 18:30:00 → 2025-12-30 18:30:00
Missing values:
date                  0
temp_max              0
temp_min              0
temp_mean             0
wind_speed_max        0
wind_speed_mean       0
humidity_mean         0
precipitation         0
visibility         3653
dtype: int64


,date,temp_max,temp_min,temp_mean,wind_speed_max,wind_speed_mean,humidity_mean,precipitation,visibility
0,2015-12-31 18:30:00,22.297501,9.297500,14.937081,9.693295,6.144559,68.453026,0.0,NaN
1,2016-01-01 18:30:00,22.047501,7.297500,14.412084,10.739832,7.835854,65.795128,0.0,NaN
2,2016-01-02 18:30:00,22.647499,8.747499,14.968333,13.854155,10.166673,62.526402,0.0,NaN
3,2016-01-03 18:30:00,22.897499,8.397500,16.064165,9.957108,6.039811,65.527191,0.0,NaN
4,2016-01-04 18:30:00,24.497499,12.497499,18.055834,8.825508,5.550406,68.818520,0.0,NaN


In [16]:
# Fix 1: Drop visibility (unavailable), fix timestamp to date only
weather_df = weather_df.drop(columns=["visibility"])
weather_df["date"] = weather_df["date"].dt.normalize()  # strips time component

# Fix 2: Trim to only dates where we have AQ data
weather_df = weather_df[
    (weather_df["date"] >= aq_daily["date"].min()) &
    (weather_df["date"] <= aq_daily["date"].max())
]

# Overwrite saved file
weather_df.to_csv(f"{DATA_DIR_RAW}/weather_daily.csv", index=False)

print(f"Weather shape after trim: {weather_df.shape}")
print(f"Range: {weather_df['date'].min()} → {weather_df['date'].max()}")
print(f"Missing values:\n{weather_df.isna().sum()}")
weather_df.head()

Weather shape after trim: (3624, 8)
Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
Missing values:
date               0
temp_max           0
temp_min           0
temp_mean          0
wind_speed_max     0
wind_speed_mean    0
humidity_mean      0
precipitation      0
dtype: int64


,date,temp_max,temp_min,temp_mean,wind_speed_max,wind_speed_mean,humidity_mean,precipitation
29,2016-01-29,25.497499,13.747499,19.370419,8.396570,5.723620,72.746208,0.5
30,2016-01-30,22.147499,12.847500,17.372501,14.007655,10.711514,77.328606,0.0
31,2016-01-31,22.097500,9.697500,15.164165,12.496718,10.017783,64.969994,0.0
32,2016-02-01,20.947500,7.447500,14.197501,15.349684,9.648987,58.050755,0.0
33,2016-02-02,21.547501,8.347500,14.468334,20.620804,12.373805,53.275467,0.0


#### Get Google Trends from PyTrends

In [ ]:
from pytrends.request import TrendReq
from dateutil.relativedelta import relativedelta
from datetime import timedelta
import time

trends_client = TrendReq(hl="en-IN", tz=330, timeout=(10, 25))

def fetch_trends_daily(keyword, start, end):
    all_frames = []
    chunk_start = datetime.strptime(start, "%Y-%m-%d")
    chunk_end   = datetime.strptime(end,   "%Y-%m-%d")

    while chunk_start < chunk_end:
        chunk_stop = min(chunk_start + timedelta(days=75), chunk_end)
        timeframe  = f"{chunk_start.strftime('%Y-%m-%d')} {chunk_stop.strftime('%Y-%m-%d')}"

        try:
            trends_client.build_payload([keyword], geo="IN-DL", timeframe=timeframe)
            df = trends_client.interest_over_time()
            if not df.empty and keyword in df.columns:
                all_frames.append(df[[keyword]])
                print(f"  ✅ {chunk_start.date()} → {chunk_stop.date()} : {len(df)} days")
            else:
                print(f"  ⚠️ No data: {chunk_start.date()} → {chunk_stop.date()}")
            time.sleep(1)
        except Exception as e:
            print(f"  ❌ {chunk_start.date()}: {e}")
            time.sleep(5)  # longer wait on error

        chunk_start = chunk_stop

    return pd.concat(all_frames) if all_frames else pd.DataFrame()


keywords = [
    "air pollution Delhi",
    "N95 mask",
    "air purifier",
    "breathing problem",
    "AQI Delhi"
]

trends_frames = []

for kw in keywords:
    print(f"\nFetching: {kw}")
    df = fetch_trends_daily(kw, START_DATE, END_DATE)
    if not df.empty:
        df = df.rename(columns={kw: kw.replace(" ", "_")})
        trends_frames.append(df)
        print(f"  ✅ Total: {len(df)} days")
    else:
        print(f"  ❌ No data for {kw}")



In [21]:
# Fix duplicate indices in each frame before concat
trends_frames_clean = [df[~df.index.duplicated(keep="first")] for df in trends_frames]

trends_df = pd.concat(trends_frames_clean, axis=1).reset_index()
trends_df.rename(columns={"index": "date"}, inplace=True)
trends_df["date"] = pd.to_datetime(trends_df["date"])
trends_df = trends_df.drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True)

trends_df.to_csv(f"{DATA_DIR_RAW}/google_trends_daily.csv", index=False)

print(f"✅ Google Trends daily shape: {trends_df.shape}")
print(f"📅 Range: {trends_df['date'].min()} → {trends_df['date'].max()}")
print(f"Missing:\n{trends_df.isna().sum()}")
trends_df.head()

✅ Google Trends daily shape: (3653, 6)
📅 Range: 2016-01-01 00:00:00 → 2025-12-31 00:00:00
Missing:
date                      0
air_pollution_Delhi     668
N95_mask               1341
air_purifier              0
breathing_problem      1863
AQI_Delhi               448
dtype: int64


,date,air_pollution_Delhi,N95_mask,air_purifier,breathing_problem,AQI_Delhi
0,2016-01-01,77.0,NaN,49,NaN,0.0
1,2016-01-02,65.0,NaN,48,NaN,0.0
2,2016-01-03,0.0,NaN,0,NaN,0.0
3,2016-01-04,100.0,NaN,0,NaN,0.0
4,2016-01-05,92.0,NaN,0,NaN,0.0


#### Data Cleaning and Preprocessing

In [26]:
import pandas as pd
import numpy as np

# Load all
aq_raw     = pd.read_csv(f"{DATA_DIR_RAW}/openaq_raw.csv")
aq_daily   = pd.read_csv(f"{DATA_DIR_RAW}/air_quality_daily.csv")
weather_df = pd.read_csv(f"{DATA_DIR_RAW}/weather_daily.csv")
trends_df  = pd.read_csv(f"{DATA_DIR_RAW}/google_trends_daily.csv")

# Parse dates
aq_raw["date"]     = pd.to_datetime(aq_raw["date"])
aq_daily["date"]   = pd.to_datetime(aq_daily["date"])
weather_df["date"] = pd.to_datetime(weather_df["date"])
trends_df["date"]  = pd.to_datetime(trends_df["date"])


/var/folders/3p/mw9w9dr17s5c0wzjh5hm1j340000gn/T/ipykernel_21217/4155857657.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  aq_raw["date"]     = pd.to_datetime(aq_raw["date"])


In [27]:
print(f"Rows before: {len(aq_raw)}")
print(f"Missing values:\n{aq_raw[aq_raw['value'].isna()][['date','location_name','parameter']]}")

# Impute 4 missing values with mean of same sensor + parameter
aq_raw["value"] = aq_raw.groupby(["sensor_id", "parameter"])["value"].transform(
    lambda x: x.fillna(x.mean())
)

print(f"\nMissing after imputation: {aq_raw['value'].isna().sum()}")

# Drop physically impossible values
print(f"\nOutlier check — PM2.5 > 1000: {(aq_raw[aq_raw['parameter']=='pm25']['value'] > 1000).sum()}")
print(f"Outlier check — PM10  > 1500: {(aq_raw[aq_raw['parameter']=='pm10']['value'] > 1500).sum()}")
print(f"Negative values: {(aq_raw['value'] < 0).sum()}")

aq_raw = aq_raw[(aq_raw["value"] >= 0)]
aq_raw.loc[aq_raw["parameter"] == "pm25", "value"] = aq_raw.loc[aq_raw["parameter"] == "pm25", "value"].clip(upper=1000)
aq_raw.loc[aq_raw["parameter"] == "pm10", "value"] = aq_raw.loc[aq_raw["parameter"] == "pm10", "value"].clip(upper=1500)

print(f"Rows after: {len(aq_raw)}")
aq_raw.to_csv(f"{DATA_DIR_PROCESSED}/openaq_raw.csv", index=False)
print("✅ openaq_raw cleaned")

Rows before: 23969
Missing values:
           date                  location_name parameter
5284 2016-03-18  US Diplomatic Post: New Delhi      pm25
5285 2016-03-19  US Diplomatic Post: New Delhi      pm25
5390 2016-02-07  US Diplomatic Post: New Delhi      pm25
5391 2016-03-07  US Diplomatic Post: New Delhi      pm25

Missing after imputation: 0

Outlier check — PM2.5 > 1000: 6
Outlier check — PM10  > 1500: 5
Negative values: 35
Rows after: 23934
✅ openaq_raw cleaned


In [28]:
print(f"Missing pm10 before: {aq_daily['pm10_avg'].isna().sum()}")
print(f"Missing pm25 before: {aq_daily['pm25_avg'].isna().sum()}")

# Check if missing pm10 days have pm25 — i.e. are they early 2016 days?
missing_pm10 = aq_daily[aq_daily["pm10_avg"].isna()]
print(f"\nMissing pm10 date range: {missing_pm10['date'].min()} → {missing_pm10['date'].max()}")

# Interpolate — linear is fine for 41 scattered gaps in a continuous signal
aq_daily["pm10_avg"] = aq_daily["pm10_avg"].interpolate(method="linear", limit=7)

# For any remaining gaps > 7 consecutive days, use seasonal median (same month)
aq_daily["month"] = aq_daily["date"].dt.month
monthly_pm10 = aq_daily.groupby("month")["pm10_avg"].median()
aq_daily["pm10_avg"] = aq_daily.apply(
    lambda row: monthly_pm10[row["month"]] if pd.isna(row["pm10_avg"]) else row["pm10_avg"],
    axis=1
)
aq_daily = aq_daily.drop(columns=["month"])

print(f"\nMissing pm10 after:  {aq_daily['pm10_avg'].isna().sum()}")
print(f"Missing pm25 after:  {aq_daily['pm25_avg'].isna().sum()}")

Missing pm10 before: 56
Missing pm25 before: 1

Missing pm10 date range: 2016-01-29 00:00:00 → 2021-05-18 00:00:00

Missing pm10 after:  0
Missing pm25 after:  1


In [29]:
# India CPCB AQI breakpoints for PM2.5
def compute_aqi_pm25(pm25):
    if pd.isna(pm25):
        return np.nan
    breakpoints = [
        (0,   30,   0,   50),
        (30,  60,   51,  100),
        (60,  90,   101, 200),
        (90,  120,  201, 300),
        (120, 250,  301, 400),
        (250, 500,  401, 500),
    ]
    for bp_lo, bp_hi, aqi_lo, aqi_hi in breakpoints:
        if bp_lo <= pm25 <= bp_hi:
            return round(((aqi_hi - aqi_lo) / (bp_hi - bp_lo)) * (pm25 - bp_lo) + aqi_lo)
    return 500  # cap

aq_daily["aqi"] = aq_daily["pm25_avg"].apply(compute_aqi_pm25)

# AQI category labels
def aqi_category(aqi):
    if pd.isna(aqi):   return np.nan
    if aqi <= 50:      return "Good"
    if aqi <= 100:     return "Satisfactory"
    if aqi <= 200:     return "Moderate"
    if aqi <= 300:     return "Poor"
    if aqi <= 400:     return "Very Poor"
    return "Severe"

aq_daily["aqi_category"] = aq_daily["aqi"].apply(aqi_category)

print("AQI distribution:")
print(aq_daily["aqi_category"].value_counts())
print(f"\nAQI range: {aq_daily['aqi'].min():.0f} → {aq_daily['aqi'].max():.0f}")
aq_daily.to_csv(f"{DATA_DIR_PROCESSED}/air_quality_daily.csv", index=False)
print("✅ AQI computed and saved")

AQI distribution:
aqi_category
Satisfactory    590
Very Poor       570
Moderate        407
Poor            277
Good            254
Severe          134
Name: count, dtype: int64

AQI range: 4 → 500
✅ AQI computed and saved


In [30]:
print(f"Weather rows before trim: {len(weather_df)}")

aq_start = aq_daily["date"].min()
aq_end   = aq_daily["date"].max()

weather_df = weather_df[
    (weather_df["date"] >= aq_start) &
    (weather_df["date"] <= aq_end)
].reset_index(drop=True)

print(f"Weather rows after trim:  {len(weather_df)}")
print(f"📅 Range: {weather_df['date'].min()} → {weather_df['date'].max()}")

# Round to 2 decimal places — Open-Meteo has excessive precision
for col in ["temp_max","temp_min","temp_mean","wind_speed_max","wind_speed_mean","humidity_mean","precipitation"]:
    weather_df[col] = weather_df[col].round(2)

weather_df.to_csv(f"{DATA_DIR_PROCESSED}/weather_daily.csv", index=False)
print("✅ Weather trimmed and saved")

Weather rows before trim: 3624
Weather rows after trim:  3624
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
✅ Weather trimmed and saved


In [34]:
def parse_cpcb_wide(filepath, year):
    df = pd.read_excel(filepath)
    df = df.rename(columns={"Day": "day"})
    
    month_map = {
        "January":1,"February":2,"March":3,"April":4,
        "May":5,"June":6,"July":7,"August":8,
        "September":9,"October":10,"November":11,"December":12
    }
    
    df_long = df.melt(id_vars="day", var_name="month_name", value_name="aqi_cpcb")
    df_long["month"] = df_long["month_name"].map(month_map)
    df_long["year"]  = year
    df_long["day"]   = pd.to_numeric(df_long["day"], errors="coerce")
    df_long["aqi_cpcb"] = pd.to_numeric(df_long["aqi_cpcb"], errors="coerce")
    
    # Drop invalid rows (NaN days, NaN values, Feb 30 etc.)
    df_long = df_long.dropna(subset=["day", "month", "aqi_cpcb"])
    df_long["date"] = pd.to_datetime(df_long[["year","month","day"]], errors="coerce")
    df_long = df_long.dropna(subset=["date"])
    
    return df_long[["date", "aqi_cpcb"]].sort_values("date").reset_index(drop=True)

# Parse both files — update filenames if different
cpcb_2023 = parse_cpcb_wide(f"{DATA_DIR}/AQI_daily_city_level_delhi_2023_delhi_2023.xlsx", 2023)
cpcb_2024 = parse_cpcb_wide(f"{DATA_DIR}/AQI_daily_city_level_delhi_2024_delhi_2024.xlsx", 2024)

cpcb_df = pd.concat([cpcb_2023, cpcb_2024], ignore_index=True)

print(f"✅ CPCB data shape: {cpcb_df.shape}")
print(f"📅 Range: {cpcb_df['date'].min()} → {cpcb_df['date'].max()}")
print(f"Missing values: {cpcb_df['aqi_cpcb'].isna().sum()}")
cpcb_df.head()

✅ CPCB data shape: (731, 2)
📅 Range: 2023-01-01 00:00:00 → 2024-12-31 00:00:00
Missing values: 0


,date,aqi_cpcb
0,2023-01-01,259.0
1,2023-01-02,357.0
2,2023-01-03,385.0
3,2023-01-04,343.0
4,2023-01-05,340.0


In [36]:
# Build CPCB rows to append to aq_daily
cpcb_rows = cpcb_df.rename(columns={"aqi_cpcb": "aqi"}).copy()
cpcb_rows["pm25_avg"]    = None
cpcb_rows["pm10_avg"]    = None
cpcb_rows["aqi_category"] = cpcb_rows["aqi"].apply(aqi_category)
cpcb_rows["aqi_source"]  = "cpcb_direct"

# Tag existing aq_daily rows
aq_daily["aqi_source"] = "openaq_computed"

# Combine — only add CPCB dates not already in aq_daily
existing_dates = set(aq_daily["date"])
cpcb_new       = cpcb_rows[~cpcb_rows["date"].isin(existing_dates)]

aq_extended = pd.concat([aq_daily, cpcb_new], ignore_index=True)
aq_extended = aq_extended.sort_values("date").reset_index(drop=True)

print(f"aq_daily rows:    {len(aq_daily)}")
print(f"CPCB new rows:    {len(cpcb_new)}")
print(f"aq_extended rows: {len(aq_extended)}")
print(f"📅 Range: {aq_extended['date'].min()} → {aq_extended['date'].max()}")

# Now build master from extended AQ
master = aq_extended.copy()
master = master.merge(weather_df,   on="date", how="left")
master = master.merge(trends_df, on="date", how="left")
master = master.sort_values("date").reset_index(drop=True)

master["year"]   = master["date"].dt.year
master["month"]  = master["date"].dt.month
master["season"] = master["month"].map({
    12: "Winter",  1: "Winter",   2: "Winter",
    3:  "Spring",  4: "Spring",   5: "Spring",
    6:  "Monsoon", 7: "Monsoon",  8: "Monsoon",
    9:  "Post-Monsoon", 10: "Post-Monsoon", 11: "Post-Monsoon"
})

master.to_csv(f"{DATA_DIR}/master_daily.csv", index=False)

print(f"\n✅ Master shape: {master.shape}")
print(f"📅 Range: {master['date'].min()} → {master['date'].max()}")
print(f"\nRows per year:")
print(master.groupby("year")["date"].count())
print(f"\nMissing values:\n{master.isna().sum()}")
master.head()

aq_daily rows:    2233
CPCB new rows:    731
aq_extended rows: 2964
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00

✅ Master shape: (2964, 21)
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00

Rows per year:
year
2016    338
2017    345
2018    220
2019    178
2020    361
2021    285
2022    191
2023    365
2024    366
2025    315
Name: date, dtype: int64

Missing values:
date                      0
pm10_avg                731
pm25_avg                732
aqi                       1
aqi_category              1
aqi_source                0
temp_max                  0
temp_min                  0
temp_mean                 0
wind_speed_max            0
wind_speed_mean           0
humidity_mean             0
precipitation             0
air_pollution_Delhi     536
N95_mask               1021
air_purifier              0
breathing_problem      1434
AQI_Delhi               428
year                      0
month                     0
season                    0
dtype: int64


/var/folders/3p/mw9w9dr17s5c0wzjh5hm1j340000gn/T/ipykernel_21217/2819954371.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  aq_extended = pd.concat([aq_daily, cpcb_new], ignore_index=True)


,date,pm10_avg,pm25_avg,aqi,aqi_category,aqi_source,temp_max,temp_min,temp_mean,wind_speed_max,...,humidity_mean,precipitation,air_pollution_Delhi,N95_mask,air_purifier,breathing_problem,AQI_Delhi,year,month,season
0,2016-01-29,267.583333,356.0,443.0,Severe,openaq_computed,25.50,13.75,19.37,8.40,...,72.75,0.5,0.0,NaN,0,NaN,0.0,2016,1,Winter
1,2016-01-30,267.583333,203.0,364.0,Very Poor,openaq_computed,22.15,12.85,17.37,14.01,...,77.33,0.0,0.0,NaN,0,NaN,0.0,2016,1,Winter
2,2016-01-31,267.583333,124.0,304.0,Very Poor,openaq_computed,22.10,9.70,15.16,12.50,...,64.97,0.0,0.0,NaN,0,NaN,0.0,2016,1,Winter
3,2016-02-01,245.000000,136.0,313.0,Very Poor,openaq_computed,20.95,7.45,14.20,15.35,...,58.05,0.0,0.0,NaN,67,NaN,0.0,2016,2,Winter
4,2016-02-02,245.000000,148.0,322.0,Very Poor,openaq_computed,21.55,8.35,14.47,20.62,...,53.28,0.0,0.0,NaN,0,NaN,0.0,2016,2,Winter


#### Visualizations

In [41]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
import warnings
warnings.filterwarnings("ignore")

master = pd.read_csv(f"{DATA_DIR}/master_daily.csv")
master["date"] = pd.to_datetime(master["date"])

print(f"✅ Master loaded: {master.shape}")
print(master.dtypes)

✅ Master loaded: (2964, 21)
date                   datetime64[ns]
pm10_avg                      float64
pm25_avg                      float64
aqi                           float64
aqi_category                   object
aqi_source                     object
temp_max                      float64
temp_min                      float64
temp_mean                     float64
wind_speed_max                float64
wind_speed_mean               float64
humidity_mean                 float64
precipitation                 float64
air_pollution_Delhi           float64
N95_mask                      float64
air_purifier                    int64
breathing_problem             float64
AQI_Delhi                     float64
year                            int64
month                           int64
season                         object
dtype: object


In [42]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [73]:
master_sorted = master.sort_values("date")
master_sorted["pm25_30d_rolling"] = master_sorted["pm25_avg"].rolling(30, min_periods=1).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=master_sorted["date"],
    y=master_sorted["pm25_avg"],
    mode="lines",
    name="Daily PM2.5",
    line=dict(color="rgba(231,76,60,0.3)", width=1),
))

fig.add_trace(go.Scatter(
    x=master_sorted["date"],
    y=master_sorted["pm25_30d_rolling"],
    mode="lines",
    name="30-Day Rolling Mean",
    line=dict(color="#c0392b", width=2.5),
))

# WHO guideline
fig.add_hline(y=15,  line_dash="dash", line_color="green",  annotation_text="WHO 24hr Guideline (15 µg/m³)")
fig.add_hline(y=60,  line_dash="dash", line_color="orange", annotation_text="India NAAQS (60 µg/m³)")

fig.update_layout(
    title="PM2.5 Daily Levels in Delhi (2016–2025) with 30-Day Trend",
    xaxis_title="Date",
    yaxis_title="PM2.5 (µg/m³)",
    height=450,
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99)
)

fig.show()

fig.write_html(f"{FIGURES_DIR}/01_pm25_trend.html")


In [74]:
heatmap_data = master.groupby(["year", "month"])["aqi"].mean().reset_index()
heatmap_pivot = heatmap_data.pivot(index="year", columns="month", values="aqi")

month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig = px.imshow(
    heatmap_pivot,
    labels=dict(x="Month", y="Year", color="Avg AQI"),
    x=month_labels,
    y=heatmap_pivot.index.tolist(),
    color_continuous_scale=[
        [0,   "#2ecc71"],
        [0.2, "#f1c40f"],
        [0.4, "#e67e22"],
        [0.6, "#e74c3c"],
        [1.0, "#7b241c"],
    ],
    aspect="auto",
    text_auto=".0f"
)

fig.update_layout(
    title="Monthly Average AQI Heatmap — Delhi (2016–2025)",
    height=420,
    xaxis_title="",
    yaxis_title=""
)
fig.update_xaxes(side="top")
fig.show()

fig.write_html(f"{FIGURES_DIR}/02_aqi_heatmap.html")


In [75]:
season_order = ["Winter", "Spring", "Monsoon", "Post-Monsoon"]
season_colors = {
    "Winter":       "#2980b9",
    "Spring":       "#27ae60",
    "Monsoon":      "#16a085",
    "Post-Monsoon": "#e67e22"
}

fig = go.Figure()

for season in season_order:
    subset = master[master["season"] == season]["aqi"].dropna()
    fig.add_trace(go.Violin(
        y=subset,
        name=season,
        box_visible=True,
        meanline_visible=True,
        fillcolor=season_colors[season],
        opacity=0.7,
        line_color="black",
        points="outliers"
    ))

fig.add_hline(y=300, line_dash="dash", line_color="red", annotation_text="'Very Poor' Threshold (300)")

fig.update_layout(
    title="AQI Distribution by Season — Delhi",
    yaxis_title="AQI",
    height=500,
    showlegend=True
)
fig.show()
fig.write_html(f"{FIGURES_DIR}/03_season_violin.html")


In [49]:
# Bin wind speed for clearer pattern
master["wind_bin"] = pd.cut(
    master["wind_speed_mean"],
    bins=[0, 2, 4, 6, 8, 12, 30],
    labels=["0-2", "2-4", "4-6", "6-8", "8-12", "12+"]
)

fig = px.scatter(
    master.dropna(subset=["wind_speed_mean", "pm25_avg"]),
    x="wind_speed_mean",
    y="pm25_avg",
    color="season",
    opacity=0.5,
    trendline="lowess",
    color_discrete_map=season_colors,
    labels={
        "wind_speed_mean": "Mean Wind Speed (km/h)",
        "pm25_avg":        "PM2.5 (µg/m³)"
    },
    title="Wind Speed vs PM2.5 — Does Wind Disperse Delhi's Pollution?"
)

fig.update_layout(height=480)
fig.show()



In [50]:
# Temp range = max - min; small range = inversion conditions
master["temp_range"] = master["temp_max"] - master["temp_min"]

fig = px.scatter(
    master.dropna(subset=["temp_range", "aqi"]),
    x="temp_range",
    y="aqi",
    color="season",
    opacity=0.4,
    trendline="lowess",
    color_discrete_map=season_colors,
    labels={
        "temp_range": "Daily Temp Range °C (Max - Min)",
        "aqi":        "AQI"
    },
    title="Temperature Inversion Proxy vs AQI — Narrow Temp Range = Trapped Pollution"
)

fig.update_layout(height=480)
fig.show()

In [76]:
# Count days per year in each AQI category
severe_days = (
    master[master["aqi_category"].isin(["Very Poor", "Severe"])]
    .groupby(["year", "aqi_category"])
    .size()
    .reset_index(name="days")
)

fig = px.bar(
    severe_days,
    x="year",
    y="days",
    color="aqi_category",
    barmode="stack",
    color_discrete_map={"Very Poor": "#e67e22", "Severe": "#7b241c"},
    labels={"days": "Number of Days", "year": "Year"},
    title="Annual Count of 'Very Poor' and 'Severe' AQI Days in Delhi",
    text="days"
)

fig.update_layout(height=450)
fig.show()

fig.write_html(f"{FIGURES_DIR}/06_severe_days.html")

In [77]:
import plotly.figure_factory as ff

corr_cols = ["pm25_avg", "pm10_avg", "aqi", "temp_mean", "temp_range",
             "wind_speed_mean", "humidity_mean", "precipitation"]

corr_matrix = master[corr_cols].dropna().corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale="RdBu_r",
    zmid=0,
    text=corr_matrix.values,
    texttemplate="%{text}",
    textfont={"size": 11},
    colorbar=dict(title="Pearson r")
))

fig.update_layout(
    title="Correlation Matrix — Pollution vs Meteorological Variables",
    height=520,
    width=650
)
fig.show()
fig.write_html(f"{FIGURES_DIR}/07_correlation_heatmap.html")


In [78]:
master_sorted = master.sort_values("date")

# Monthly average for cleaner signal
monthly = master_sorted.resample("ME", on="date").agg({
    "pm25_avg":            "mean",
    "air_pollution_Delhi": "mean",
    "AQI_Delhi":           "mean"
}).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=monthly["date"],
    y=monthly["pm25_avg"],
    name="Monthly Avg PM2.5",
    line=dict(color="#e74c3c", width=2)
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=monthly["date"],
    y=monthly["air_pollution_Delhi"],
    name="Search: 'air pollution Delhi'",
    line=dict(color="#2980b9", width=2, dash="dot")
), secondary_y=True)

fig.add_trace(go.Scatter(
    x=monthly["date"],
    y=monthly["AQI_Delhi"],
    name="Search: 'AQI Delhi'",
    line=dict(color="#8e44ad", width=2, dash="dash")
), secondary_y=True)

fig.update_layout(
    title="Public Search Interest vs Actual PM2.5 Levels — Does Awareness Follow Pollution?",
    height=480,
    hovermode="x unified"
)
fig.update_yaxes(title_text="PM2.5 (µg/m³)",       secondary_y=False)
fig.update_yaxes(title_text="Google Trends Index",  secondary_y=True)

fig.show()
fig.write_html(f"{FIGURES_DIR}/08_trends_vs_pm25.html")


In [79]:
cat_order  = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
cat_colors = {
    "Good":         "#2ecc71",
    "Satisfactory": "#a9dfbf",
    "Moderate":     "#f39c12",
    "Poor":         "#e67e22",
    "Very Poor":    "#e74c3c",
    "Severe":       "#7b241c"
}

yearly_cats = (
    master.groupby(["year", "aqi_category"])
    .size()
    .reset_index(name="days")
)
yearly_cats["aqi_category"] = pd.Categorical(yearly_cats["aqi_category"], categories=cat_order, ordered=True)
yearly_cats = yearly_cats.sort_values(["year", "aqi_category"])

fig = px.bar(
    yearly_cats,
    x="year",
    y="days",
    color="aqi_category",
    barmode="stack",
    color_discrete_map=cat_colors,
    category_orders={"aqi_category": cat_order},
    labels={"days": "Number of Days", "year": "Year"},
    title="Annual AQI Category Breakdown — Delhi's Air Quality Profile by Year"
)

fig.update_layout(height=480)
fig.show()

fig.write_html(f"{FIGURES_DIR}/09_aqi_stacked_bar.html")


In [84]:
# Ratio > 0.5 indicates combustion/fine particles (vehicles, burning)
# Ratio < 0.5 indicates coarse particles (dust, construction)
master_ratio = master.dropna(subset=["pm25_avg", "pm10_avg"]).copy()
master_ratio = master_ratio[master_ratio["pm10_avg"] > 0]
master_ratio["pm_ratio"] = master_ratio["pm25_avg"] / master_ratio["pm10_avg"]
master_ratio["pm_ratio"] = master_ratio["pm_ratio"].clip(0, 1.5)

monthly_ratio = master_ratio.groupby(["year", "month"])["pm_ratio"].mean().reset_index()

fig = px.line(
    monthly_ratio,
    x="month",
    y="pm_ratio",
    color="year",
    markers=True,
    labels={"pm_ratio": "PM2.5 / PM10 Ratio", "month": "Month"},
    title="Monthly PM2.5/PM10 Ratio by Year — Combustion vs Dust Signature"
)

fig.add_hline(y=0.5, line_dash="dash", line_color="red",
              annotation_text="0.5 threshold (combustion vs dust)")

fig.update_layout(
    height=480,
    xaxis=dict(tickmode="array", tickvals=list(range(1,13)),
               ticktext=["Jan","Feb","Mar","Apr","May","Jun",
                         "Jul","Aug","Sep","Oct","Nov","Dec"])
)

fig.show()
fig.write_html(f"{FIGURES_DIR}/11_pm_ratio.html")


In [85]:
# Find rain days and track AQI for 5 days before and after
rain_days = master[master["precipitation"] > 5]["date"].values

before_after = []
for rain_date in rain_days:
    for offset in range(-5, 6):
        target = pd.Timestamp(rain_date) + pd.Timedelta(days=offset)
        row    = master[master["date"] == target]
        if not row.empty:
            before_after.append({
                "offset":        offset,
                "aqi":           row["aqi"].values[0],
                "precipitation": row["precipitation"].values[0]
            })

ba_df    = pd.DataFrame(before_after)
ba_agg   = ba_df.groupby("offset")["aqi"].agg(["mean","std"]).reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ba_agg["offset"],
    y=ba_agg["mean"],
    mode="lines+markers",
    name="Mean AQI",
    line=dict(color="#e74c3c", width=2.5),
    error_y=dict(type="data", array=ba_agg["std"], visible=True, color="rgba(231,76,60,0.3)")
))

fig.add_vline(x=0, line_dash="dash", line_color="blue",
              annotation_text="Rain Day")

fig.update_layout(
    title="AQI Before and After Significant Rain Events (>5mm) — Pollution Recovery Curve",
    xaxis_title="Days Relative to Rain Event",
    yaxis_title="Mean AQI",
    height=450,
    xaxis=dict(tickmode="linear", dtick=1)
)

fig.show()
fig.write_html(f"{FIGURES_DIR}/12_rain_recovery.html")

In [86]:
master["day_of_week"] = master["date"].dt.dayofweek  # 0=Mon, 6=Sun
master["is_weekend"]  = master["day_of_week"].isin([5, 6])
master["day_name"]    = master["date"].dt.day_name()

day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

day_aqi = master.groupby("day_name")["aqi"].agg(["mean","median","std"]).reindex(day_order).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=day_aqi["day_name"],
    y=day_aqi["mean"],
    name="Mean AQI",
    marker_color=["#e74c3c"]*5 + ["#2980b9"]*2,
    error_y=dict(type="data", array=day_aqi["std"], visible=True)
))

fig.add_trace(go.Scatter(
    x=day_aqi["day_name"],
    y=day_aqi["median"],
    mode="lines+markers",
    name="Median AQI",
    line=dict(color="#2c3e50", width=2, dash="dot")
))

fig.update_layout(
    title="Weekday vs Weekend AQI — Assessing Traffic's Role in Delhi's Pollution",
    yaxis_title="AQI",
    height=450
)

fig.show()
fig.write_html(f"{FIGURES_DIR}/13_weekday_weekend.html")

In [87]:
from scipy.stats import pearsonr

# Test correlation between AQI and search interest at different time lags
lag_cols    = ["air_pollution_Delhi", "AQI_Delhi", "N95_mask"]
lag_results = []

master_clean = master.dropna(subset=["aqi"] + lag_cols).sort_values("date")

for col in lag_cols:
    for lag in range(-14, 15):  # -14 to +14 days
        shifted = master_clean["aqi"].shift(lag)
        valid   = master_clean[col].notna() & shifted.notna()
        if valid.sum() > 100:
            r, p = pearsonr(master_clean.loc[valid, col], shifted[valid])
            lag_results.append({"keyword": col, "lag": lag, "correlation": r, "p_value": p})

lag_df = pd.DataFrame(lag_results)

fig = px.line(
    lag_df,
    x="lag",
    y="correlation",
    color="keyword",
    markers=False,
    labels={"lag": "Lag (days) — negative = search leads AQI", "correlation": "Pearson r"},
    title="Cross-Correlation: Google Search Interest vs AQI at Different Time Lags"
)

fig.add_vline(x=0,  line_dash="dash", line_color="black", annotation_text="Same day")
fig.add_hline(y=0,  line_dash="dot",  line_color="gray")

fig.update_layout(height=480)
fig.show()

fig.write_html(f"{FIGURES_DIR}/15_lag_correlation.html")


In [60]:
from scipy import stats as scipy_stats

pm25_clean = master["pm25_avg"].dropna()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("PM2.5 Distribution (with log-normal fit)", "QQ Plot — Normality Check")
)

# Left: histogram + log-normal fit
fig.add_trace(go.Histogram(
    x=pm25_clean,
    nbinsx=60,
    name="PM2.5",
    marker_color="rgba(231,76,60,0.6)",
    histnorm="probability density"
), row=1, col=1)

x_range = np.linspace(pm25_clean.min(), pm25_clean.max(), 200)
mu, std  = pm25_clean.mean(), pm25_clean.std()
log_mu   = np.log(pm25_clean).mean()
log_std  = np.log(pm25_clean).std()
pdf_vals = scipy_stats.lognorm.pdf(x_range, s=log_std, scale=np.exp(log_mu))

fig.add_trace(go.Scatter(
    x=x_range, y=pdf_vals,
    name="Log-Normal Fit",
    line=dict(color="#2c3e50", width=2)
), row=1, col=1)

# Right: QQ plot
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(pm25_clean)
fig.add_trace(go.Scatter(
    x=osm, y=osr,
    mode="markers",
    name="QQ Points",
    marker=dict(color="rgba(231,76,60,0.5)", size=3)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=[min(osm), max(osm)],
    y=[slope*min(osm)+intercept, slope*max(osm)+intercept],
    mode="lines",
    name="Normal Reference",
    line=dict(color="black", width=1.5)
), row=1, col=2)

fig.update_layout(
    title="PM2.5 Statistical Distribution Diagnostics",
    height=440,
    showlegend=True
)
fig.update_xaxes(title_text="PM2.5 (µg/m³)",      row=1, col=1)
fig.update_xaxes(title_text="Theoretical Quantiles", row=1, col=2)
fig.update_yaxes(title_text="Density",             row=1, col=1)
fig.update_yaxes(title_text="Sample Quantiles",    row=1, col=2)

fig.show()

# Print stats
skew = pm25_clean.skew()
kurt = pm25_clean.kurtosis()
_, p_norm = scipy_stats.normaltest(pm25_clean)



In [61]:
summary_cols = ["pm25_avg", "pm10_avg", "aqi", "temp_mean",
                "wind_speed_mean", "humidity_mean", "precipitation"]

summary = master[summary_cols].describe().T
summary["skewness"] = master[summary_cols].skew().round(3)
summary["missing"]  = master[summary_cols].isna().sum()
summary = summary.round(2)

print("📊 Summary Statistics — Master Dataset")
print(summary.to_string())

📊 Summary Statistics — Master Dataset
                  count    mean     std    min     25%     50%     75%     max  skewness  missing
pm25_avg         2232.0  104.45   84.51   2.50   42.69   78.78  140.22  750.83      2.01      732
pm10_avg         2233.0  222.49  133.66  -5.67  111.29  204.17  310.00  952.00      0.86      731
aqi              2963.0  197.25  124.36   4.00   81.00  173.00  313.00  500.00      0.36        1
temp_mean        2964.0   24.81    6.81   6.66   19.37   26.92   29.77   39.19     -0.49        0
wind_speed_mean  2964.0    9.19    3.30   2.05    6.79    8.63   10.97   22.97      0.90        0
humidity_mean    2964.0   61.86   19.02  10.22   49.96   64.80   77.45   95.72     -0.59        0
precipitation    2964.0    2.22    7.07   0.00    0.00    0.00    0.90  122.70      7.92        0
